# Classical prediction, modern ensembles and honest uncertainty

LDA/QDA, neighbors, kernels and linear penalties remain useful modeling choices. Compare them with bagging, extra trees, adaptive/histogram/native boosting, voting and cross-validated stacking. The 243 selected models share isolated fitting, selection and calibration roles; candidate and compute budgets differ. Regression includes geographic shift.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd() if Path("configs").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))


## Probability scores and data roles

A binary linear SVM returns a margin; use two ordered logits before temperature calibration. Temperature and conformal scores need distinct calibration data. A test threshold slider explores outcomes; it cannot retrospectively select a deployable threshold.

In [2]:
from scipy.special import softmax
from sklearn.datasets import make_classification
from sklearn.svm import LinearSVC
from learning_lab.classical import logits
x, y = make_classification(n_samples=120, n_features=5, random_state=8)
svm = LinearSVC(dual="auto").fit(x[:80], y[:80])
scores = logits(svm, x[80:])
assert np.array_equal(scores.argmax(1), svm.predict(x[80:]))
pd.DataFrame(softmax(scores[:5], axis=1), columns=["class0", "class1"])

,class0,class1
0,0.801691,0.198309
1,0.968046,0.031954
2,0.348745,0.651255
3,0.347709,0.652291
4,0.907321,0.092679


## All fitted families

Log-loss measures probability quality, macro-F1 weights classes equally, and accuracy depends on the decision rule. Rankings need not agree. The bundled diagnostic datasets do not establish clinical readiness.

In [3]:
fits = pd.read_csv(root / "docs/classical_individual_fits.csv")
assert len(fits) == 243
fits.groupby(["case", "model"])[["accuracy", "log_loss", "rmse"]].mean().dropna(how="all").round(4)

accuracy  log_loss    rmse
case           model                                    
adult          adaboost         0.8608    0.3027     NaN
               bagging          0.8603    0.3127     NaN
               catboost         0.8725    0.2797     NaN
               extra_trees      0.8495    0.3268     NaN
               hist_boosting    0.8714    0.2795     NaN
...                                ...       ...     ...
housing_random rbf_svr             NaN       NaN  0.5577
               ridge               NaN       NaN  0.7313
               stacking            NaN       NaN  0.4683
               tree                NaN       NaN  0.6552
               voting              NaN       NaN  0.5006

[81 rows x 3 columns]

## Geographic uncertainty

Absolute-residual conformal intervals target 90% coverage under exchangeability. The geographic test can violate that assumption. Report coverage and width together, and preserve failures.

In [4]:
housing = fits[fits.case.str.startswith("housing")]
summary = housing.groupby(["case", "model"])[["rmse", "interval_coverage", "interval_width"]].mean()
summary.loc[(slice(None), ["ridge", "catboost", "stacking"]), :]

,,rmse,interval_coverage,interval_width
case,model,,,
housing_geographic,ridge,0.659662,0.937291,2.191961
housing_random,ridge,0.731315,0.897933,2.134967
housing_geographic,catboost,0.894555,0.489628,1.355303
housing_random,catboost,0.464116,0.899871,1.378873
housing_geographic,stacking,0.860618,0.530637,1.392763
housing_random,stacking,0.468319,0.895753,1.384170


## Stacking without preprocessing leakage

Each base estimator owns its preprocessing pipeline. The outer fitting partition contains the inner folds. Scaling and imputation refit inside those folds; out-of-fold predictions train the final estimator. Selection labels and calibration labels are never inner-fold fitting data.

In [5]:
protocol = json.loads((root / "results/reference/classical/digits/0/metrics.json").read_text())
stack = next(row for row in protocol["rows"] if row["model"] == "stacking")
{key: stack[key] for key in ["candidate_count", "selected_candidate", "validation_macro_f1", "seconds_including_selection"]}

{'candidate_count': 2,
 'selected_candidate': 0,
 'validation_macro_f1': 0.9785145888594163,
 'seconds_including_selection': 16.528576981276274}

Reproduce with `python scripts/run_classical.py --dataset digits --seed 0`; verify every public prediction with `python scripts/validate_classical.py`. Read [classical results](../docs/classical_results.md), and open [the explorer](../docs/explorer.html) for confusion matrices, threshold controls and interval plots.